# COMP6011 Task 2 — Experiment 2: Prompt Robustness Analysis

This experiment evaluates the impact of prompt design on model performance and output reliability.

The study compares three prompting strategies:
- Baseline prompting  
- Few-shot prompting  
- Strict structured prompting  

The goal is to determine how prompt design influences classification accuracy and consistency.

## 1. Hardware Check

This cell checks GPU availability before loading large language models.

In [ ]:
!nvidia-smi

Thu Apr  9 10:46:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Environment Setup

This cell installs the required packages for model loading, inference, data processing, and evaluation.

In [ ]:
!pip install -q transformers accelerate bitsandbytes sentencepiece pandas openpyxl scikit-learn huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.2 MB/s eta 0:00:00


## 3. Hugging Face Authentication

A Hugging Face token is required to access the selected open-weight models. The token is entered at runtime and is not stored in the notebook.

In [ ]:
from getpass import getpass
HF_TOKEN = getpass("Enter your Hugging Face token: ")

Enter your Hugging Face token: ··········


## 4. Dataset Upload

The dataset is manually uploaded into the notebook. It contains conversational cases labelled using the five risk categories: safe, indicator, ideation, behavior, and attempt.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving student_assignment_10_cases.xlsx to student_assignment_10_cases.xlsx


## 5. Core Functions and Evaluation Setup

This cell defines the labels, helper functions, model loading process, output extraction method, and evaluation metrics.

These functions are reused across all prompt strategies and models to ensure consistent evaluation.

In [ ]:
import os
import gc
import json
import re
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

FILE_PATH = "student_assignment_10_cases.xlsx"
SHEET_NAME = "Assignment_Cases"

LABELS = ["safe", "indicator", "ideation", "behavior", "attempt"]
QWEN_MODEL = "Qwen/Qwen2.5-7B-Instruct"
MISTRAL_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"

df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)
df.columns = [c.strip() for c in df.columns]

def normalize_label(label):
    if pd.isna(label):
        return None
    label = str(label).strip().lower()
    mapping = {"behaviour": "behavior"}
    return mapping.get(label, label)

df["true_label"] = df["Risk Level"].apply(normalize_label)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

def load_model(model_name):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        token=HF_TOKEN,
        trust_remote_code=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=HF_TOKEN,
        quantization_config=bnb_config,
        device_map="auto",
        max_memory={0: "13GiB", "cpu": "30GiB"},
        offload_folder="/content/offload",
        trust_remote_code=True
    )
    return tokenizer, model

def clear_gpu(*objects):
    for obj in objects:
        try:
            del obj
        except:
            pass
    gc.collect()
    torch.cuda.empty_cache()

## 6. Prompt Design Strategies

This cell defines the three prompt templates used in the experiment.

The baseline prompt provides a simple classification instruction, the strict prompt enforces JSON formatting, and the few-shot prompt includes example classifications to guide model behaviour.

In [ ]:
BASELINE_PROMPT = '''
Classify the following dialogue into one of these suicide risk categories:
safe, indicator, ideation, behavior, attempt.

Return the label and a short reason.
'''

STRICT_JSON_PROMPT = '''
You are a strict classification system.

Task:
Classify the dialogue into EXACTLY ONE of these labels:
safe, indicator, ideation, behavior, attempt

Output MUST be valid JSON ONLY.

Format:
{"label": "<one label>", "reason": "<brief explanation>"}

Rules:
- Output only JSON
- No markdown
- No extra text
- Label must be exactly one of the five allowed labels
'''

FEW_SHOT_PROMPT = '''
You are a strict suicide risk classification assistant.

Classify each dialogue into exactly one label:
safe, indicator, ideation, behavior, attempt

Return valid JSON only:
{"label": "<one label>", "reason": "<brief explanation>"}

Example 1:
Dialogue:
User: I had a rough day, but I know things will get better.
AI: I'm glad you're holding on. Is there anything helping you cope right now?
Output:
{"label": "safe", "reason": "The user expresses distress but no suicide-related warning signs."}

Example 2:
Dialogue:
User: Sometimes I feel like disappearing and not waking up.
AI: That sounds really painful. Have these thoughts become more frequent recently?
Output:
{"label": "ideation", "reason": "The dialogue suggests suicidal thoughts without clear evidence of action."}

Now classify the next dialogue.
'''

def build_prompt(dialogue: str, mode: str = "baseline") -> str:
    if mode == "baseline":
        return f"""{BASELINE_PROMPT}

Dialogue:
{dialogue}
""".strip()
    elif mode == "strict":
        return f"""{STRICT_JSON_PROMPT}

Dialogue:
{dialogue}
""".strip()
    elif mode == "fewshot":
        return f"""{FEW_SHOT_PROMPT}

Dialogue:
{dialogue}
""".strip()
    else:
        raise ValueError("mode must be one of: baseline, strict, fewshot")

def extract_json_label(text: str):
    if not text:
        return None, None, text

    match = re.search(r'\{.*?\}', text, re.DOTALL)
    if match:
        try:
            obj = json.loads(match.group(0))
            label = normalize_label(obj.get("label"))
            reason = obj.get("reason", "")
            if label in LABELS:
                return label, reason, text
        except Exception:
            pass

    text_lower = text.lower()
    for label in LABELS:
        if label in text_lower:
            return label, None, text

    return None, None, text

def classify_dialogue_local(tokenizer, model, dialogue: str, mode: str, max_new_tokens: int = 120):
    prompt = build_prompt(dialogue, mode=mode)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    generated_text = full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text

    label, reason, raw = extract_json_label(generated_text)
    return {"label": label, "reason": reason, "raw_output": raw}

## 7. Prompt Experiment Function

This cell defines the function used to run all prompt strategies on each dialogue case.

For each model, the function records the prompt mode, predicted label, reasoning, and raw output.

In [ ]:
def run_prompt_experiment(model_name, tokenizer, model, df):
    rows = []
    for mode in ["baseline", "strict", "fewshot"]:
        for _, row in df.iterrows():
            out = classify_dialogue_local(tokenizer, model, row["Paraphrased Dialogue"], mode=mode)
            rows.append({
                "Model": model_name,
                "Prompt Mode": mode,
                "Case ID": row["Case ID"],
                "true_label": row["true_label"],
                "pred_label": out["label"],
                "reason": out["reason"],
                "raw_output": out["raw_output"]
            })
    return pd.DataFrame(rows)

def evaluate_prompt_results(results_df):
    summary_rows = []
    for (model_name, prompt_mode), group in results_df.groupby(["Model", "Prompt Mode"]):
        total = len(group)
        valid_group = group[group["pred_label"].isin(LABELS)]
        valid = len(valid_group)
        valid_rate = valid / total if total > 0 else 0.0

        if valid > 0:
            y_true = valid_group["true_label"].tolist()
            y_pred = valid_group["pred_label"].tolist()
            acc = accuracy_score(y_true, y_pred)
            precision, recall, f1, _ = precision_recall_fscore_support(
                y_true, y_pred, labels=LABELS, average="macro", zero_division=0
            )
        else:
            acc = precision = recall = f1 = 0.0

        summary_rows.append({
            "Model": model_name,
            "Prompt Mode": prompt_mode,
            "Valid Outputs": f"{valid}/{total}",
            "Valid Rate": round(valid_rate, 2),
            "Accuracy": round(acc, 2),
            "Macro Precision": round(precision, 2),
            "Macro Recall": round(recall, 2),
            "Macro F1": round(f1, 2)
        })

    return pd.DataFrame(summary_rows)

## 8. Prompt Evaluation — Qwen2.5 7B Instruct

This section evaluates Qwen2.5 across baseline, strict, and few-shot prompt settings.

In [ ]:
qwen_tokenizer, qwen_model = load_model(QWEN_MODEL)
qwen_prompt_results = run_prompt_experiment("Qwen2.5 7B", qwen_tokenizer, qwen_model, df)
qwen_prompt_summary = evaluate_prompt_results(qwen_prompt_results)

display(qwen_prompt_summary)

qwen_prompt_results.to_csv("qwen_prompt_experiment_cases.csv", index=False)
qwen_prompt_summary.to_csv("qwen_prompt_experiment_summary.csv", index=False)
print("Saved Qwen prompt experiment files.")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


,Model,Prompt Mode,Valid Outputs,Valid Rate,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Qwen2.5 7B,baseline,9/10,0.9,0.33,0.17,0.3,0.22
1,Qwen2.5 7B,fewshot,10/10,1.0,0.40,0.31,0.4,0.34
2,Qwen2.5 7B,strict,10/10,1.0,0.20,0.14,0.2,0.16


Saved Qwen prompt experiment files.


## 9. Clear GPU Memory After Qwen

The Qwen model is removed from memory before loading the next model to reduce the risk of out-of-memory errors.

In [ ]:
clear_gpu(qwen_model, qwen_tokenizer)
!nvidia-smi

Thu Apr  9 10:31:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P0             43W /   70W |    5601MiB /  15360MiB |     17%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 10. Prompt Evaluation — Mistral 7B Instruct

This section evaluates Mistral across the same three prompt settings so that results can be compared with Qwen under consistent conditions.

In [ ]:
mistral_tokenizer, mistral_model = load_model(MISTRAL_MODEL)
mistral_prompt_results = run_prompt_experiment("Mistral 7B", mistral_tokenizer, mistral_model, df)
mistral_prompt_summary = evaluate_prompt_results(mistral_prompt_results)

display(mistral_prompt_summary)

mistral_prompt_results.to_csv("mistral_prompt_experiment_cases.csv", index=False)
mistral_prompt_summary.to_csv("mistral_prompt_experiment_summary.csv", index=False)
print("Saved Mistral prompt experiment files.")

config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

,Model,Prompt Mode,Valid Outputs,Valid Rate,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Mistral 7B,baseline,9/10,0.9,0.44,0.26,0.4,0.29
1,Mistral 7B,fewshot,9/10,0.9,0.56,0.41,0.5,0.41
2,Mistral 7B,strict,5/10,0.5,0.20,0.04,0.2,0.07


Saved Mistral prompt experiment files.


## 11. Clear GPU Memory After Mistral

The Mistral model is removed from memory after inference to keep the notebook environment clean.

In [ ]:
clear_gpu(mistral_model, mistral_tokenizer)
!nvidia-smi

Thu Apr  9 10:55:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             46W /   70W |    4301MiB /  15360MiB |     75%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 12. Results Summary and Export

This section combines the prompt experiment summaries from Qwen and Mistral, displays the final results table, and exports the outputs for use in the report and appendix.

In [ ]:
summary_frames = []
if 'qwen_prompt_summary' in globals():
    summary_frames.append(qwen_prompt_summary)
if 'mistral_prompt_summary' in globals():
    summary_frames.append(mistral_prompt_summary)

if summary_frames:
    all_prompt_summary = pd.concat(summary_frames, ignore_index=True)
    display(all_prompt_summary)
    all_prompt_summary.to_csv("all_prompt_experiment_summary.csv", index=False)
    print("Saved all_prompt_experiment_summary.csv")
else:
    print("No summaries available yet.")

## 13. Key Observations

The results show that prompt design has a clear effect on model performance and output reliability.

Few-shot prompting produced the strongest overall results across both models, while strict JSON prompting improved format control but reduced classification performance. This suggests that prompt engineering is an important factor in safety-critical classification tasks, alongside model selection.